In [0]:
from pyspark.sql import functions as F

fake_date = '2026-08-29'
sample = spark.table('bronze.patient').limit(5)
modified = (
    sample
    .withColumn('resource_json', F.col('resource_json').withField('gender', F.lit('unknown')))
    .withColumn('bronze_load_date', F.lit(fake_date))
    .withColumn('ingestion_ts', F.current_timestamp())
)
modified.write.format('delta').mode('append').saveAsTable('bronze.patient')
print('Inserted', modified.count(), 'modified rows with bronze_load_date =', fake_date)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

SILVER_DB = "silver"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")

def apply_scd2(resource_name, bronze_load_date):
    source_table = f"bronze.{resource_name.lower()}"
    target_table = f"{SILVER_DB}.{resource_name.lower()}"

    bronze_df = spark.table(source_table).filter(F.col("bronze_load_date") == bronze_load_date)

    incoming = (
        bronze_df
        .withColumn("row_hash", F.sha2(F.to_json("resource_json"), 256))
        .dropDuplicates(["resource_id", "row_hash"])
        .withColumn("valid_from", F.current_timestamp())
        .withColumn("valid_to", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
    )

    if not spark.catalog.tableExists(target_table):
        incoming.write.format("delta").mode("overwrite").saveAsTable(target_table)
        print(f"[{resource_name}] Initial Silver load: {incoming.count()} rows")
        return

    silver = DeltaTable.forName(spark, target_table)

    current_silver = (
        silver.toDF().filter("is_current = true")
        .select("resource_id", F.col("row_hash").alias("existing_hash"))
    )
    changes = (
        incoming.join(current_silver, "resource_id", "left")
        .filter(
            F.col("existing_hash").isNull() |
            (F.col("row_hash") != F.col("existing_hash"))
        )
    )

    change_count = changes.count()
    if change_count == 0:
        print(f"[{resource_name}] No changes detected.")
        return

    (
        silver.alias("s")
        .merge(
            changes.select("resource_id").distinct().alias("c"),
            "s.resource_id = c.resource_id AND s.is_current = true",
        )
        .whenMatchedUpdate(set={
            "is_current": "false",
            "valid_to": "current_timestamp()",
        })
        .execute()
    )

    #changes.write.format("delta").mode("append").saveAsTable(target_table)
    changes.drop("existing_hash").write.format("delta").mode("append").saveAsTable(target_table)
    print(f"[{resource_name}] {change_count} new/changed versions inserted.")

from datetime import datetime, UTC
import json
from pathlib import Path

# Load resource list from config (no hardcoded lists)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root = str(Path(notebook_path).parent.parent)
with open(f"/Workspace{project_root}/config/resources.json", "r") as f:
    config = json.load(f)

extraction_date = datetime.now(UTC).strftime("%Y-%m-%d")

for resource in config["resources"]:
    apply_scd2(resource, extraction_date)

In [0]:
%sql
-- Confirm valid_from is populated, valid_to is null, is_current is true for all rows (expected on a first run)
SELECT resource_id, valid_from, valid_to, is_current 
FROM silver.patient 
LIMIT 10

In [0]:
spark.sql("SELECT resource_id, valid_from, valid_to, is_current FROM silver.patient WHERE is_current = false LIMIT 10").show(truncate=False)

In [0]:
spark.sql("SELECT DISTINCT bronze_load_date FROM bronze.patient").show()

In [0]:
spark.sql("SELECT resource_id, valid_from, is_current FROM silver.patient WHERE is_current = true AND resource_id IN ('137584287', '137584285', '137584306', '137584286', '137584300')").show()

In [0]:
spark.sql("SELECT resource_id, COUNT(*) as version_count FROM silver.patient WHERE resource_id IN ('137584287', '137584285', '137584306', '137584286', '137584300') GROUP BY resource_id").show()